# Notebook 02 — Experiment 1 latency analysis

Canonical analysis entry point for **Experiment 1: MCP-overhead comparison across Cells A / B / C**.

Until all three cells have committed captures, this notebook runs in **preflight mode**: it checks the expected artifact layout (per `docs/runbook.md` §3.4 and `#25`) and writes an availability snapshot to `results/metrics/`. When the captures land, the same notebook rolls forward into the real latency analysis without a structural rewrite.

There is one useful intermediate milestone before full A / B / C readiness: once `#104` lands the first shared **Cell B** AaT artifact, this notebook should validate that Cell B already satisfies the Experiment 1 / Experiment 2 shared-anchor contract (scenario IDs, trial indices, latency rows, and dual-use metadata), even though the headline overhead deltas still wait on Cells A and C.

**Artifact layout expected** (from `scripts/run_experiment.sh`):

```
benchmarks/cell_<A|B|C>_<name>/
├── config.json       # canonical reproducibility config + wandb_run_url
├── summary.json      # aggregate stats: latency_seconds_{mean,p50,p95}, mcp_latency_*, tool_*, etc.
└── raw/<run-id>/
    ├── meta.json             # run metadata + profiling linkage (#27)
    ├── latencies.jsonl       # one row per scenario×trial
    ├── harness.log
    ├── vllm.log              # if LAUNCH_VLLM=1
    └── <date>_<cell>_<model>_<orch>_<mcp>_<scenario>_run<n>.json
```

Run IDs are `<slurm_job_id>_<EXPERIMENT_NAME>` (Slurm) or `local-<YYYYMMDD-HHMMSS>_<EXPERIMENT_NAME>` (local).

**Latest-run selection:** picks the run dir with the highest `meta.json.started_at` (ISO 8601). Falls back to directory mtime if no run has `started_at`. Lexicographic sort on run IDs was rejected because local and Slurm IDs don't interleave chronologically.


In [1]:
import json
import platform
import re
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "benchmarks").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path")

REPO_ROOT = find_repo_root(Path.cwd())
BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_METRICS_DIR = REPO_ROOT / "results" / "metrics"
RESULTS_FIGURES_DIR = REPO_ROOT / "results" / "figures"
RESULTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Declare every output path up front so degraded/zero-cell branches can replace
# (not silently leave) stale prior-run artifacts.
AVAILABILITY_CSV = RESULTS_METRICS_DIR / "notebook02_cell_availability.preflight.csv"
CONTRACT_CSV = RESULTS_METRICS_DIR / "notebook02_cell_b_contract.preflight.csv"
LATENCY_SUMMARY_CSV = RESULTS_METRICS_DIR / "notebook02_latency_summary.csv"
OVERHEAD_CSV = RESULTS_METRICS_DIR / "notebook02_mcp_overhead.csv"
LATENCY_FIGURE_PNG = RESULTS_FIGURES_DIR / "notebook02_latency_comparison.png"


def _write_deferred_csv(path: Path, columns: list[str], note: str) -> None:
    """Replace any stale CSV at `path` with a schema-only frame plus a note row.

    Downstream readers can detect the deferred state via `note` instead of reading
    real-looking rows from a previous run.
    """
    note_row = {col: "" for col in columns}
    if "note" not in note_row:
        columns = [*columns, "note"]
        note_row = {**note_row, "note": note}
    else:
        note_row["note"] = note
    pd.DataFrame([note_row], columns=columns).to_csv(path, index=False)


print(f"Repo root:   {REPO_ROOT}")
print(f"Python:      {platform.python_version()}")
print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")

Repo root:   /Users/wax/coding/hpml-assetopsbench-smart-grid-mcp/.codex/worktrees/codex-fnd-exp-26-32-34-staging
Python:      3.12.8
pandas:      3.0.0
numpy:       2.2.6
matplotlib:  3.10.8


## Cells scanned

Notebook 02 is scoped to the three Experiment 1 conditions. Cells Y and Z belong to Experiment 2 and live in Notebook 03.

| Cell | Label | Directory |
|---|---|---|
| A | Direct | `benchmarks/cell_A_direct/` |
| B | MCP baseline | `benchmarks/cell_B_mcp_baseline/` |
| C | MCP optimized | `benchmarks/cell_C_mcp_optimized/` |

Per-cell config lives at the canonical `configs/aat_{direct,mcp_baseline,mcp_optimized}.env` paths on `main`.

Cell B is intentionally shared with Experiment 2. That means the first real AaT artifact from `#104` is already useful here as a contract check even before A and C are ready enough for the actual overhead deltas.


In [3]:
CELL_SPECS = [
    {"cell": "A", "label": "Direct",        "path": BENCHMARKS_DIR / "cell_A_direct"},
    {"cell": "B", "label": "MCP baseline",  "path": BENCHMARKS_DIR / "cell_B_mcp_baseline"},
    {"cell": "C", "label": "MCP optimized", "path": BENCHMARKS_DIR / "cell_C_mcp_optimized"},
]

SUMMARY_FIELDS = [
    # orchestration / identity
    "experiment_family", "experiment_cell", "orchestration_mode", "mcp_mode",
    "scenario_set_name", "scenario_set_hash", "model_id", "model_provider",
    "serving_stack", "quantization_mode",
    # outcome
    "run_status", "scenarios_attempted", "scenarios_completed",
    "success_rate", "failure_count",
    # latency
    "wall_clock_seconds_total",
    "latency_seconds_mean", "latency_seconds_p50", "latency_seconds_p95",
    # tool / MCP
    "tool_call_count_total", "tool_call_count_mean", "tool_error_count",
    "mcp_latency_seconds_mean", "mcp_latency_seconds_p95",
    "tool_latency_seconds_mean",
    # wandb / judge (may be null in Experiment 1)
    "wandb_run_url",
    "judge_score_mean", "judge_score_p50", "judge_score_p95",
    "judge_pass_rate",
]

META_PROFILING_FIELDS = ["profiling_dir", "profiling_artifact", "profiling_summary"]
EXPECTED_SHARED_B_CONTRIBUTING_EXPERIMENTS = frozenset({"exp1_mcp_overhead", "exp2_orchestration"})
EXPECTED_LATENCY_COLUMNS = frozenset({"scenario_file", "trial_index", "latency_seconds", "output_path"})


def _load_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def _load_jsonl(path: Path) -> tuple[list[dict], list[dict]]:
    """Read a JSONL file. Returns (rows, parse_errors).

    Each parse_error is `{"path": str(path), "line_number": int, "raw": str, "reason": str}`.
    Caller decides how loud to be — `_count_jsonl_rows` ignores errors for the
    cheap availability count, but the Cell B contract validator surfaces them
    as an explicit failing check so corrupt artifacts cannot hide downstream.
    """
    rows: list[dict] = []
    errors: list[dict] = []
    if path is None or not path.exists():
        return rows, errors
    with path.open() as handle:
        for lineno, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                errors.append({
                    "path": str(path),
                    "line_number": lineno,
                    "raw": stripped[:200],
                    "reason": str(exc),
                })
    return rows, errors


def _count_jsonl_rows(path: Path) -> int:
    rows, _ = _load_jsonl(path)
    return len(rows)


def _parse_iso_ts(ts: str | None):
    """Return a timezone-aware datetime from an ISO-8601 string, or None.

    Handles the `Z` suffix by converting to `+00:00` before parsing. If the
    string is missing a timezone offset entirely, it is treated as naive and
    returned as-is — downstream code must not mix naive and aware datetimes.
    """
    if not ts:
        return None
    try:
        normalized = ts.replace("Z", "+00:00") if ts.endswith("Z") else ts
        return datetime.fromisoformat(normalized)
    except (TypeError, ValueError):
        return None


def _latest_run_dir(raw_dir: Path) -> Path | None:
    """Pick the latest run dir by parsed `meta.json.started_at`; fall back to mtime.

    Raw string comparison on ISO-8601 is not chronological across mixed offsets
    (e.g. `2026-04-21T00:00:00-04:00` > `2026-04-21T00:00:00Z` lexicographically
    but NOT chronologically), so we parse into timezone-aware datetimes first.
    Entries with unparseable or missing timestamps fall back to directory mtime.
    """
    if not raw_dir.exists():
        return None
    candidates = [p for p in raw_dir.iterdir() if p.is_dir()]
    if not candidates:
        return None

    with_ts, without_ts = [], []
    for p in candidates:
        meta_path = p / "meta.json"
        parsed = None
        if meta_path.exists():
            meta = _load_json(meta_path)
            if meta:
                parsed = _parse_iso_ts(meta.get("started_at") or meta.get("finished_at"))
        if parsed is not None:
            with_ts.append((parsed, p))
        else:
            without_ts.append((p.stat().st_mtime, p))

    if with_ts:
        return max(with_ts, key=lambda pair: pair[0])[1]
    return max(without_ts, key=lambda pair: pair[0])[1]


def _resolve_repo_path(raw_path: str | None) -> Path | None:
    if not raw_path:
        return None
    path = Path(raw_path)
    return path if path.is_absolute() else REPO_ROOT / path


def _scenario_jsons(run_dir: Path) -> list[Path]:
    if not run_dir.exists():
        return []
    return sorted(
        p for p in run_dir.glob("*.json")
        if p.name not in {"meta.json", "summary.json", "config.json"}
    )


def scan_cell(cell_spec: dict) -> dict:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()]) if raw_dir.exists() else []
    latest_run = _latest_run_dir(raw_dir)

    summary = _load_json(cell_dir / "summary.json")
    config = _load_json(cell_dir / "config.json")
    latencies_path = latest_run / "latencies.jsonl" if latest_run else None
    meta = _load_json(latest_run / "meta.json") if latest_run else None

    row = {
        "cell": cell_spec["cell"],
        "label": cell_spec["label"],
        "cell_dir": str(cell_dir.relative_to(REPO_ROOT)),
        "config_present": (cell_dir / "config.json").exists(),
        "summary_present": (cell_dir / "summary.json").exists(),
        "raw_run_count": len(run_dirs),
        "latest_run_id": latest_run.name if latest_run else "",
        "latency_rows": _count_jsonl_rows(latencies_path) if latencies_path else 0,
    }
    for field in SUMMARY_FIELDS:
        row[field] = summary.get(field) if summary else None
    # Profiling linkage (#27) stamps these onto meta.json for the linked run.
    for field in META_PROFILING_FIELDS:
        row[field] = meta.get(field) if meta else None
    return row


availability_df = pd.DataFrame(scan_cell(spec) for spec in CELL_SPECS)
availability_df


,cell,label,cell_dir,config_present,summary_present,raw_run_count,latest_run_id,latency_rows,experiment_family,experiment_cell,...,mcp_latency_seconds_p95,tool_latency_seconds_mean,wandb_run_url,judge_score_mean,judge_score_p50,judge_score_p95,judge_pass_rate,profiling_dir,profiling_artifact,profiling_summary
0,A,Direct,benchmarks/cell_A_direct,True,True,1,8979314_aat_direct,6,exp1_mcp_overhead,A,...,None,None,https://wandb.ai/assetopsbench-smartgrid/asset...,None,None,None,None,profiling/traces/8979314_cell_a,profiling-vq976ljq,{'profiling/gpu_util_mean': 16.890140845070423...
1,B,MCP baseline,benchmarks/cell_B_mcp_baseline,True,True,1,8979314_aat_mcp_baseline,6,exp1_mcp_overhead,B,...,None,None,https://wandb.ai/assetopsbench-smartgrid/asset...,None,None,None,None,profiling/traces/8979314_cell_b,profiling-qejvnoug,{'profiling/gpu_util_mean': 24.156626506024097...
2,C,MCP optimized,benchmarks/cell_C_mcp_optimized,False,False,0,,0,NaN,NaN,...,None,None,NaN,None,None,None,None,NaN,NaN,None


In [4]:
# Write the cheap availability snapshot so downstream consumers don't have
# to rerun the scan just to see what exists.
availability_df.to_csv(AVAILABILITY_CSV, index=False)
print(f"Wrote {AVAILABILITY_CSV.relative_to(REPO_ROOT)}")

display_cols = [
    "cell", "label", "config_present", "summary_present", "raw_run_count",
    "latest_run_id", "latency_rows", "run_status",
    "orchestration_mode", "mcp_mode",
    "latency_seconds_mean", "latency_seconds_p95",
    "mcp_latency_seconds_mean", "tool_error_count",
    "profiling_dir", "wandb_run_url",
]
display(availability_df[display_cols])

Wrote results/metrics/notebook02_cell_availability.preflight.csv


,cell,label,config_present,summary_present,raw_run_count,latest_run_id,latency_rows,run_status,orchestration_mode,mcp_mode,latency_seconds_mean,latency_seconds_p95,mcp_latency_seconds_mean,tool_error_count,profiling_dir,wandb_run_url
0,A,Direct,True,True,1,8979314_aat_direct,6,success,agent_as_tool,direct,12.187792,18.574839,None,0.0,profiling/traces/8979314_cell_a,https://wandb.ai/assetopsbench-smartgrid/asset...
1,B,MCP baseline,True,True,1,8979314_aat_mcp_baseline,6,success,agent_as_tool,baseline,13.383503,16.654413,None,0.0,profiling/traces/8979314_cell_b,https://wandb.ai/assetopsbench-smartgrid/asset...
2,C,MCP optimized,False,False,0,,0,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN


## Readiness gate

Notebook 02 distinguishes two readiness modes:

- **Full readiness** — all three cells (A, B, C) have committed `config.json`,
  `summary.json`, and at least one raw run with latency rows. The full MCP
  overhead decomposition (B−A, B−C, C−A) is reported.
- **Partial readiness** — at least one cell is ready. Whatever pairwise overhead
  can be computed from the present cells is reported and labelled
  `(partial — pending C)` or similar. The notebook still produces a usable
  paper figure and CSV from the data on disk.

The shared Cell B contract check fires as soon as the first real B run lands,
regardless of full readiness, because B is shared with Experiment 2.


In [5]:
availability_df["analysis_ready"] = (
    availability_df["config_present"]
    & availability_df["summary_present"]
    & (availability_df["raw_run_count"] > 0)
    & (availability_df["latency_rows"] > 0)
)

ready_cells = availability_df.loc[availability_df["analysis_ready"], "cell"].tolist()
missing_cells = availability_df.loc[~availability_df["analysis_ready"], "cell"].tolist()
ready_set = set(ready_cells)

print(f"Ready cells:   {ready_cells if ready_cells else 'none'}")
print(f"Missing cells: {missing_cells if missing_cells else 'none'}")

analysis_ready = len(missing_cells) == 0
partial_ready = len(ready_cells) > 0 and not analysis_ready

if analysis_ready:
    print("\nFull-readiness mode: all of A/B/C present. Reporting full overhead decomposition.")
elif partial_ready:
    print(f"\nPartial-readiness mode: {ready_cells} present. "
          "Reporting pairwise overhead deltas for available cell pairs; "
          "missing pairs deferred until remaining cells land.")
else:
    print("\nScaffold mode: waiting on real captures for Cells A, B, C.")
    print("Expected captures per #25 come from scripts/run_experiment.sh runs against configs/aat_*.env.")

if "B" in ready_set and not analysis_ready:
    print("Shared Cell B note: the AaT anchor is present, so #104 / Experiment 2 contract checks "
          "can start now even though the full A/B/C overhead table is incomplete.")


Ready cells:   ['A', 'B']
Missing cells: ['C']

Partial-readiness mode: ['A', 'B'] present. Reporting pairwise overhead deltas for available cell pairs; missing pairs deferred until remaining cells land.
Shared Cell B note: the AaT anchor is present, so #104 / Experiment 2 contract checks can start now even though the full A/B/C overhead table is incomplete.


## Shared Cell B contract

Cell B is shared between Experiment 1 and Experiment 2. As soon as the first real AaT artifact lands, validate that the latest B run already exposes the downstream join contract cleanly: dual-use metadata in `config.json` / `summary.json`, canonical scenario IDs in the raw outputs, and filename-aligned `trial_index` values in `latencies.jsonl`.


In [6]:
CONTRACT_CSV_COLUMNS = ["check", "ok", "detail"]


def validate_shared_cell_b_contract(cell_spec: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    cell_dir = cell_spec["path"]
    latest_run = _latest_run_dir(cell_dir / "raw")
    if latest_run is None:
        return pd.DataFrame(), pd.DataFrame()

    config = _load_json(cell_dir / "config.json") or {}
    summary = _load_json(cell_dir / "summary.json") or {}
    latency_rows, latency_parse_errors = _load_jsonl(latest_run / "latencies.jsonl")
    detail_rows = []

    for idx, row in enumerate(latency_rows, start=1):
        row_keys = set(row)
        output_path = _resolve_repo_path(row.get("output_path"))
        source_path = _resolve_repo_path(row.get("scenario_file"))
        output_payload = _load_json(output_path) if output_path and output_path.exists() else None
        source_payload = _load_json(source_path) if source_path and source_path.exists() else None

        output_scenario = output_payload.get("scenario") if isinstance(output_payload, dict) else None
        output_scenario_id = output_scenario.get("id") if isinstance(output_scenario, dict) else None
        source_scenario_id = source_payload.get("id") if isinstance(source_payload, dict) else None

        trial_index = row.get("trial_index")
        filename_trial_match = re.search(r"_run(\d+)$", output_path.stem) if output_path else None
        filename_trial_index = int(filename_trial_match.group(1)) if filename_trial_match else None

        detail_rows.append({
            "row_index": idx,
            "missing_columns": ", ".join(sorted(EXPECTED_LATENCY_COLUMNS - row_keys)),
            "output_path_exists": bool(output_path and output_path.exists()),
            "source_path_exists": bool(source_path and source_path.exists()),
            "canonical_scenario_id_present": bool(output_scenario_id),
            "source_scenario_id_match": bool(output_scenario_id and source_scenario_id and output_scenario_id == source_scenario_id),
            "trial_index": trial_index,
            "filename_trial_index": filename_trial_index,
            "trial_index_match": filename_trial_index == trial_index,
            "scenario_file": row.get("scenario_file"),
            "output_path": row.get("output_path"),
            "scenario_id": output_scenario_id,
        })

    detail_df = pd.DataFrame(detail_rows)
    summary_rows = []

    def add_check(name: str, ok: bool, detail: str):
        summary_rows.append({"check": name, "ok": bool(ok), "detail": detail})

    config_contrib = frozenset(config.get("contributing_experiments") or [])
    summary_contrib = frozenset(summary.get("contributing_experiments") or [])
    add_check(
        "config_contributing_experiments",
        config_contrib == EXPECTED_SHARED_B_CONTRIBUTING_EXPERIMENTS,
        f"config.json has {sorted(config_contrib)}",
    )
    add_check(
        "summary_contributing_experiments",
        summary_contrib == EXPECTED_SHARED_B_CONTRIBUTING_EXPERIMENTS,
        f"summary.json has {sorted(summary_contrib)}",
    )

    add_check(
        "latency_rows_parseable",
        len(latency_parse_errors) == 0,
        f"unparseable JSONL rows: {len(latency_parse_errors)}"
        + (f" (first: line {latency_parse_errors[0]['line_number']})" if latency_parse_errors else ""),
    )

    if detail_df.empty:
        add_check("latency_rows_present", False, "latencies.jsonl has no parseable rows")
        add_check("latency_required_columns", False, "cannot validate columns without rows")
        add_check("unique_latency_join_keys", False, "cannot validate join-key uniqueness without rows")
        add_check("canonical_scenario_ids_present", False, "cannot validate scenario IDs without rows")
        add_check("source_scenario_ids_match", False, "cannot validate source IDs without rows")
        add_check("trial_indices_match_output_filenames", False, "cannot validate trial indices without rows")
    else:
        add_check("latency_rows_present", True, f"latencies.jsonl has {len(detail_df)} row(s)")
        add_check(
            "latency_required_columns",
            (detail_df["missing_columns"] == "").all(),
            f"rows missing columns: {(detail_df['missing_columns'] != '').sum()}",
        )
        latency_pairs = detail_df[["scenario_file", "trial_index"]].copy()
        duplicate_pairs = int(latency_pairs.duplicated(keep=False).sum()) if not latency_pairs.empty else 0
        add_check(
            "unique_latency_join_keys",
            duplicate_pairs == 0,
            f"duplicate (scenario_file, trial_index) rows: {duplicate_pairs}",
        )
        add_check(
            "canonical_scenario_ids_present",
            detail_df["canonical_scenario_id_present"].all(),
            f"rows missing canonical scenario.id: {(~detail_df['canonical_scenario_id_present']).sum()}",
        )
        add_check(
            "source_scenario_ids_match",
            detail_df["source_scenario_id_match"].all(),
            f"rows with source/output id mismatch: {(~detail_df['source_scenario_id_match']).sum()}",
        )
        add_check(
            "trial_indices_match_output_filenames",
            detail_df["trial_index_match"].all(),
            f"rows with trial-index mismatch: {(~detail_df['trial_index_match']).sum()}",
        )

    return pd.DataFrame(summary_rows), detail_df


cell_b_contract_summary = pd.DataFrame()
cell_b_contract_detail = pd.DataFrame()
cell_b_row = availability_df.loc[availability_df["cell"] == "B"].iloc[0]
if int(cell_b_row["raw_run_count"] or 0) > 0:
    cell_b_contract_summary, cell_b_contract_detail = validate_shared_cell_b_contract(CELL_SPECS[1])
    if not cell_b_contract_summary.empty:
        cell_b_contract_summary.to_csv(CONTRACT_CSV, index=False)
        print(f"Wrote {CONTRACT_CSV.relative_to(REPO_ROOT)}")
        display(cell_b_contract_summary)
        failed_checks = cell_b_contract_summary.loc[~cell_b_contract_summary["ok"], "check"].tolist()
        if failed_checks:
            print(f"WARNING: shared Cell B contract gaps remain: {failed_checks}")
        else:
            print("Shared Cell B contract checks passed for the latest B run.")
    else:
        _write_deferred_csv(
            CONTRACT_CSV, CONTRACT_CSV_COLUMNS,
            "Cell B raw run present but contract validator returned no rows.",
        )
        print(f"Wrote deferred-state {CONTRACT_CSV.relative_to(REPO_ROOT)}.")
else:
    _write_deferred_csv(
        CONTRACT_CSV, CONTRACT_CSV_COLUMNS,
        "Shared Cell B contract deferred - waiting on the first raw Cell B artifact.",
    )
    print(f"Wrote deferred-state {CONTRACT_CSV.relative_to(REPO_ROOT)} (no Cell B raw run yet).")

Wrote results/metrics/notebook02_cell_b_contract.preflight.csv


,check,ok,detail
0,config_contributing_experiments,False,config.json has []
1,summary_contributing_experiments,False,summary.json has []
2,latency_rows_parseable,True,unparseable JSONL rows: 0
3,latency_rows_present,True,latencies.jsonl has 6 row(s)
4,latency_required_columns,True,rows missing columns: 0
5,unique_latency_join_keys,True,"duplicate (scenario_file, trial_index) rows: 0"
6,canonical_scenario_ids_present,False,rows missing canonical scenario.id: 6
7,source_scenario_ids_match,False,rows with source/output id mismatch: 6
8,trial_indices_match_output_filenames,True,rows with trial-index mismatch: 0


## Per-scenario latencies

Pull `latencies.jsonl` from the **latest** run in every cell that has data, join into a single dataframe, and report per-cell summary stats. Cells without captures are skipped (not blocking). The summary CSV is written whenever ≥1 cell has data so downstream consumers always have an artifact to read.


In [7]:
LATENCY_SUMMARY_COLUMNS = ["cell", "label", "count", "latency_mean", "latency_p50", "latency_p95", "latency_max"]


def load_cell_latencies(cell_spec: dict) -> pd.DataFrame:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    latest_run = _latest_run_dir(raw_dir)
    if latest_run is None:
        return pd.DataFrame()
    rows, errors = _load_jsonl(latest_run / "latencies.jsonl")
    if errors:
        print(
            f"WARNING: {cell_spec['cell']}: {len(errors)} unparseable JSONL row(s) in "
            f"{latest_run.name}/latencies.jsonl; first at line {errors[0]['line_number']}: {errors[0]['reason']}"
        )
    enriched = []
    for row in rows:
        row["cell"] = cell_spec["cell"]
        row["label"] = cell_spec["label"]
        row["run_id"] = latest_run.name
        enriched.append(row)
    return pd.DataFrame(enriched)


per_cell_dfs = []
for spec in CELL_SPECS:
    if spec["cell"] in ready_set:
        df = load_cell_latencies(spec)
        if not df.empty:
            per_cell_dfs.append(df)

if per_cell_dfs:
    latency_df = pd.concat(per_cell_dfs, ignore_index=True)
    latency_summary = (
        latency_df.groupby(["cell", "label"], as_index=False)["latency_seconds"]
        .agg(
            count="count",
            latency_mean="mean",
            latency_p50="median",
            latency_p95=lambda s: s.quantile(0.95),
            latency_max="max",
        )
        .sort_values("cell")
        .reset_index(drop=True)
    )
    latency_summary.to_csv(LATENCY_SUMMARY_CSV, index=False)
    print(f"Wrote {LATENCY_SUMMARY_CSV.relative_to(REPO_ROOT)} ({len(latency_summary)} cell(s))")
    display(latency_summary)
else:
    latency_df = pd.DataFrame()
    latency_summary = pd.DataFrame()
    _write_deferred_csv(
        LATENCY_SUMMARY_CSV, LATENCY_SUMMARY_COLUMNS,
        "No cells have latency rows yet - per-scenario aggregation deferred.",
    )
    print(f"Wrote deferred-state {LATENCY_SUMMARY_CSV.relative_to(REPO_ROOT)}.")

Wrote results/metrics/notebook02_latency_summary.csv (2 cell(s))


,cell,label,count,latency_mean,latency_p50,latency_p95,latency_max
0,A,Direct,6,12.187792,12.150448,17.294409,18.574839
1,B,MCP baseline,6,13.383503,13.086995,16.270298,16.654413


## MCP overhead decomposition

The benchmark claim the paper needs is a clean overhead decomposition:

- **MCP transport overhead** = Cell B − Cell A. The cost of the JSON-RPC layer with nothing else changed.
- **Optimization delta** = Cell B − Cell C. What the MCP optimizations (batched tool calls, caching, etc.) actually save versus the naive baseline.
- **Net overhead after optimization** = Cell C − Cell A. Ideally small; if large, MCP still has a floor that optimization can't erase.

**Paired on `(scenario_file, trial_index)`.** The math is per-pair deltas → then median/p95 of the delta distribution. Computing whole-cell medians and subtracting would produce misleading numbers if the cells cover different scenario subsets or trial counts.

**Partial-readiness behaviour.** Each delta is computed independently from whichever pair of cells is present. With Cells A and B only, the headline MCP transport overhead (B − A) is already publishable. The optimization delta and net overhead wait for Cell C and surface as `n_pairs=0` rows.


In [8]:
def _pair_set(group: pd.DataFrame) -> frozenset:
    if "scenario_file" not in group.columns or "trial_index" not in group.columns:
        return frozenset()
    return frozenset(zip(group["scenario_file"], group["trial_index"]))


# All paired metrics we know how to compute, declared once with their cell pair.
OVERHEAD_METRICS = [
    ("MCP transport overhead (B - A)",          "B", "A"),
    ("Optimization delta (B - C)",              "B", "C"),
    ("Net overhead after optimization (C - A)", "C", "A"),
]


def _empty_overhead_row(metric: str, note: str) -> dict:
    return {
        "metric": metric,
        "n_pairs": 0,
        "median_seconds": None,
        "p95_seconds": None,
        "note": note,
    }


def compute_pair_delta(latency_df: pd.DataFrame, lhs: str, rhs: str) -> tuple[pd.Series, str]:
    """Return (delta_series, note). Empty series + note if pair not computable."""
    coverage = {cell: _pair_set(g) for cell, g in latency_df.groupby("cell")}
    lhs_pairs = coverage.get(lhs, frozenset())
    rhs_pairs = coverage.get(rhs, frozenset())
    if not lhs_pairs or not rhs_pairs:
        missing = [c for c in (lhs, rhs) if not coverage.get(c)]
        return pd.Series(dtype=float), f"deferred - missing cell(s): {missing}"

    common = lhs_pairs & rhs_pairs
    if not common:
        return pd.Series(dtype=float), f"no shared (scenario, trial) pairs between {lhs} and {rhs}"

    subset = latency_df[latency_df["cell"].isin([lhs, rhs])]
    dup_mask = subset.duplicated(subset=["cell", "scenario_file", "trial_index"], keep=False)
    if bool(dup_mask.any()):
        return pd.Series(dtype=float), f"duplicate (cell, scenario, trial) rows in {lhs}/{rhs}; refusing to collapse"

    pivoted = subset.pivot_table(
        index=["scenario_file", "trial_index"],
        columns="cell",
        values="latency_seconds",
        aggfunc="first",
    )
    if lhs not in pivoted.columns or rhs not in pivoted.columns:
        return pd.Series(dtype=float), f"missing pivot column for {lhs} or {rhs}"

    common_index = pd.MultiIndex.from_tuples(sorted(common), names=["scenario_file", "trial_index"])
    paired = pivoted.loc[pivoted.index.intersection(common_index)]
    delta = (paired[lhs] - paired[rhs]).dropna()
    coverage_note = ""
    extra_lhs = len(lhs_pairs - common)
    extra_rhs = len(rhs_pairs - common)
    if extra_lhs or extra_rhs:
        coverage_note = f"restricted to {len(common)} common pairs (dropped {lhs}:{extra_lhs}, {rhs}:{extra_rhs})"
    return delta, coverage_note


if not latency_df.empty:
    overhead_rows = []
    for metric, lhs, rhs in OVERHEAD_METRICS:
        delta, note = compute_pair_delta(latency_df, lhs, rhs)
        if delta.empty:
            overhead_rows.append(_empty_overhead_row(metric, note))
        else:
            overhead_rows.append({
                "metric": metric,
                "n_pairs": int(delta.notna().sum()),
                "median_seconds": float(delta.median()),
                "p95_seconds": float(delta.quantile(0.95)),
                "note": note,
            })
    overhead = pd.DataFrame(overhead_rows)
    overhead.to_csv(OVERHEAD_CSV, index=False)
    label = "partial" if not analysis_ready else "full"
    print(f"Wrote {OVERHEAD_CSV.relative_to(REPO_ROOT)} ({label}-readiness mode)")
    display(overhead)
else:
    overhead = pd.DataFrame()
    OVERHEAD_CSV_COLUMNS = ["metric", "n_pairs", "median_seconds", "p95_seconds", "note"]
    _write_deferred_csv(
        OVERHEAD_CSV, OVERHEAD_CSV_COLUMNS,
        "No cells have latency rows yet - MCP overhead decomposition deferred.",
    )
    print(f"Wrote deferred-state {OVERHEAD_CSV.relative_to(REPO_ROOT)}.")


Wrote results/metrics/notebook02_mcp_overhead.csv (partial-readiness mode)


,metric,n_pairs,median_seconds,p95_seconds,note
0,MCP transport overhead (B - A),6,0.022992,7.561326,
1,Optimization delta (B - C),0,NaN,NaN,deferred - missing cell(s): ['C']
2,Net overhead after optimization (C - A),0,NaN,NaN,deferred - missing cell(s): ['C']


## Figure - latency by cell

Paper-facing plot. p50 bars with p95 error marks for every cell that has data, saved to `results/figures/`. Cells still pending capture are rendered as hatched placeholder bars at zero so the figure layout matches the final A/B/C deliverable when C lands.


In [9]:
cell_order = ["A", "B", "C"]
labels = []
p50_vals = []
p95_vals = []
is_present = []

if not latency_df.empty:
    summary_indexed = latency_summary.set_index("cell")
else:
    summary_indexed = pd.DataFrame().set_index(pd.Index([], name="cell"))

for c in cell_order:
    spec = next(s for s in CELL_SPECS if s["cell"] == c)
    labels.append(f"{c} - {spec['label']}")
    if c in summary_indexed.index:
        p50_vals.append(float(summary_indexed.loc[c, "latency_p50"]))
        p95_vals.append(float(summary_indexed.loc[c, "latency_p95"]))
        is_present.append(True)
    else:
        p50_vals.append(0.0)
        p95_vals.append(0.0)
        is_present.append(False)

# Visible placeholder height: 8% of the largest present p95 (or a fallback 1.0
# when no cells are ready yet) so the hatched rectangle is actually visible.
present_p95 = [v for v, present in zip(p95_vals, is_present) if present]
placeholder_height = (max(present_p95) * 0.08) if present_p95 else 1.0
plot_heights = [v if present else placeholder_height for v, present in zip(p50_vals, is_present)]

fig, ax = plt.subplots(figsize=(7, 4.5))
palette = {"A": "#4c78a8", "B": "#f58518", "C": "#54a24b"}
face_colors = [palette[c] if present else "none" for c, present in zip(cell_order, is_present)]
edge_colors = ["black" if present else "#666666" for present in is_present]
bars = ax.bar(labels, plot_heights, color=face_colors, edgecolor=edge_colors, linewidth=1.0)
for bar, present in zip(bars, is_present):
    if not present:
        bar.set_hatch("//")
for bar, mid, high, present in zip(bars, p50_vals, p95_vals, is_present):
    cx = bar.get_x() + bar.get_width() / 2
    if present:
        ax.plot([cx, cx], [mid, high], color="black", linewidth=1)
        ax.plot([cx - 0.12, cx + 0.12], [high, high], color="black", linewidth=1)
    else:
        ax.text(
            cx, placeholder_height / 2, "pending",
            ha="center", va="center", fontsize=9, color="#444444",
            rotation=0,
        )

if not any(is_present):
    title_suffix = " (zero-cell preview - all cells pending capture)"
elif all(is_present):
    title_suffix = ""
else:
    title_suffix = " (partial - hatched bars pending capture)"
ax.set_title(f"Experiment 1 - latency by cell (p50 bar, p95 cap){title_suffix}")
ax.set_ylabel("End-to-end latency (seconds)")
ax.set_xlabel("Condition")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

fig.savefig(LATENCY_FIGURE_PNG, dpi=200, bbox_inches="tight")
plt.close(fig)
mode_label = "full" if all(is_present) else ("partial" if any(is_present) else "zero-cell preview")
print(f"Wrote {LATENCY_FIGURE_PNG.relative_to(REPO_ROOT)} ({mode_label})")

Wrote results/figures/notebook02_latency_comparison.png (partial)


## Profiling linkage — visibility

`#27` (`01043c5`) wires `capture_around.sh` → `log_profiling_to_wandb.py` → benchmark WandB run, and stamps `profiling_dir` / `profiling_artifact` / `profiling_summary` back into `meta.json`. Surface those fields here so a reviewer can tell at a glance which cells have profiling attached — and which need a rerun with profiling wired in.

In [10]:
prof_cols = ["cell", "label", "latest_run_id", "profiling_dir", "profiling_artifact", "profiling_summary", "wandb_run_url"]
prof_view = availability_df[prof_cols].copy()
# Truncate URLs + long summary dicts for display only.
prof_view["profiling_summary"] = prof_view["profiling_summary"].apply(
    lambda v: "set" if isinstance(v, dict) and v else ("missing" if v is None else "empty")
)
display(prof_view)

,cell,label,latest_run_id,profiling_dir,profiling_artifact,profiling_summary,wandb_run_url
0,A,Direct,8979314_aat_direct,profiling/traces/8979314_cell_a,profiling-vq976ljq,set,https://wandb.ai/assetopsbench-smartgrid/asset...
1,B,MCP baseline,8979314_aat_mcp_baseline,profiling/traces/8979314_cell_b,profiling-qejvnoug,set,https://wandb.ai/assetopsbench-smartgrid/asset...
2,C,MCP optimized,,NaN,NaN,missing,NaN


## What changes later

When the remaining `#25` captures land, this notebook should not need a new shape. Expected forward deltas:

- when Cell C lands, the `Optimization delta (B - C)` and `Net overhead after optimization (C - A)` rows in `notebook02_mcp_overhead.csv` populate automatically alongside the already-published `MCP transport overhead (B - A)` row
- the figure swaps the hatched placeholder bar for a real C bar without any other layout changes
- split end-to-end latency from tool-only latency when `tool_latency_seconds` is populated per-row
- pull `mcp_latency_seconds` from the per-scenario JSONs when the runner surfaces per-step MCP timings
- join profiling-summary stats (GPU util, memory, power) into the comparison when profiling runs are attached
- close out the shared Cell B contract gaps surfaced above (canonical `scenario.id` propagation in run output JSONs; `contributing_experiments` populated in `config.json` / `summary.json`) so Experiment 2 can join cleanly to the same anchor

When Experiment 2 captures land but not Experiment 1, **do not** promote Experiment 2 metrics into this notebook - they belong to Notebook 03.
